# Winkansen van voetbal wedstrijden voorspellen
#### Een onderzoek van Lucas Hoetink, Hidde monsma, Khaleel Al-Aqel, Gamal Al-Sabaeei

In dit rapport gaan wij de winkansen van voetbalwedstrijden voorspellen. Dit gaan we doen op basis van principes uit de data science en doormiddel van machine learning modellen.

---

## Importing Libraries

Als eerste moeten we altijd de juiste libraries importeren zodat we toegang hebben tot de benodigde tools.

In [1]:
import sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import os
from pathlib import Path
import tempfile

---

## Loading Data

Als volgende moeten we de data ophalen en inladen uit de `SQL` database

In [2]:
subprocess.run(['pip', 'install', 'kaggle', '-q'], capture_output=True)

# Set up Kaggle credentials
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

credentials = {
    "username": "lucashoetink",
    "key": "22e34cd52b72413f58087cacec1636c7"
}

with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump(credentials, f)

os.chmod(kaggle_dir / 'kaggle.json', 0o600)

# Download to a writable temp directory
temp_dir = tempfile.gettempdir()
os.system(f'kaggle datasets download -d hugomathien/soccer --unzip -p {temp_dir}')

# Connect to the database from temp directory
db_path = os.path.join(temp_dir, 'database.sqlite')
connection = sql.connect(db_path)
print(f"Connected to database at: {db_path}")

Connected to database at: C:\Users\gamal\AppData\Local\Temp\database.sqlite


---

## SQL Exploration

In [3]:
# See all table names
cursor = connection.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables:", tables)

Available tables: [('sqlite_sequence',), ('Player_Attributes',), ('Player',), ('Match',), ('League',), ('Country',), ('Team',), ('Team_Attributes',)]


In [10]:
df = pd.read_sql("SELECT * FROM Player_Attributes ", connection)
display(df)

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
1,2,218353,505942,2015-11-19 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
2,3,218353,505942,2015-09-21 00:00:00,62.0,66.0,right,medium,medium,49.0,...,54.0,48.0,65.0,66.0,69.0,6.0,11.0,10.0,8.0,8.0
3,4,218353,505942,2015-03-20 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
4,5,218353,505942,2007-02-22 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183973,183974,102359,39902,2009-08-30 00:00:00,83.0,85.0,right,medium,low,84.0,...,88.0,83.0,22.0,31.0,30.0,9.0,20.0,84.0,20.0,20.0
183974,183975,102359,39902,2009-02-22 00:00:00,78.0,80.0,right,medium,low,74.0,...,88.0,70.0,32.0,31.0,30.0,9.0,20.0,73.0,20.0,20.0
183975,183976,102359,39902,2008-08-30 00:00:00,77.0,80.0,right,medium,low,74.0,...,88.0,70.0,32.0,31.0,30.0,9.0,20.0,73.0,20.0,20.0
183976,183977,102359,39902,2007-08-30 00:00:00,78.0,81.0,right,medium,low,74.0,...,88.0,53.0,28.0,32.0,30.0,9.0,20.0,73.0,20.0,20.0


In [5]:
# Load any table into a DataFrame
df = pd.read_sql("SELECT * FROM Country ", connection)
display(df)

,id,name
0,1,Belgium
1,1729,England
2,4769,France
3,7809,Germany
4,10257,Italy
5,13274,Netherlands
6,15722,Poland
7,17642,Portugal
8,19694,Scotland
9,21518,Spain


In [6]:
df = pd.read_sql("SELECT * FROM League ", connection)
display(df)

,id,country_id,name
0,1,1,Belgium Jupiler League
1,1729,1729,England Premier League
2,4769,4769,France Ligue 1
3,7809,7809,Germany 1. Bundesliga
4,10257,10257,Italy Serie A
5,13274,13274,Netherlands Eredivisie
6,15722,15722,Poland Ekstraklasa
7,17642,17642,Portugal Liga ZON Sagres
8,19694,19694,Scotland Premier League
9,21518,21518,Spain LIGA BBVA


| Columns            	| Beschrijving                                                         	| Relevantie 	|
|--------------------	|----------------------------------------------------------------------	|------------	|
| League(id)         	| Het eigen unieke nummer van de competitie.                           	| 1          	|
| League(country_id) 	| Het nummer dat naar het land verwijst.                               	| 1          	|
| League(name)       	| De naam zoals wij die kennen, bijvoorbeeld "Netherlands Eredivisie". 	| 1          	|
| Country(id)        	| is een uniek codenummer dat de computer gebruikt om te zoeken.       	| 1          	|
| Country(name)      	| De name is gewoon de naam zoals wij die kennen, zoals "Netherlands". 	| 1          	|

### Identificatie voor Sparta Rotterdam

In [15]:
CLUB_NAAM = 'Sparta Rotterdam'

# Basisgegevens uit de 'Team' tabel
df_team = pd.read_sql_query(f"SELECT team_api_id, team_fifa_api_id, team_short_name FROM Team WHERE team_long_name = '{CLUB_NAAM}'", connection)

if not df_team.empty:
    # We halen de waardes uit de eerste rij van het resultaat
    t_api_id = df_team.iloc[0]['team_api_id']
    t_fifa_id = df_team.iloc[0]['team_fifa_api_id']
    t_short = df_team.iloc[0]['team_short_name']
    
    # ompetitie ID's uit de 'Match' tabel 
    df_match = pd.read_sql_query(f"SELECT country_id, league_id FROM Match WHERE home_team_api_id = {t_api_id} LIMIT 1", connection)
    l_id = df_match.iloc[0]['league_id']
    c_id = df_match.iloc[0]['country_id']

    # Attribuut ID uit 'Team_Attributes' 
    df_attr = pd.read_sql_query(f"SELECT id FROM Team_Attributes WHERE team_api_id = {t_api_id} LIMIT 1", connection)
    attr_id = df_attr.iloc[0]['id'] if not df_attr.empty else "Niet gevonden"

    # We zetten alles in een simpel overzicht
    all_ids = {
        "Club Naam": CLUB_NAAM,
        "Short Name": t_short,
        "Team API ID": t_api_id,
        "Team FIFA ID": t_fifa_id,
        "League ID": l_id,
        "Country ID": c_id,
        "Attribute ID": attr_id
    }


    print("Gevonden identifiers:")
    for naam, waarde in all_ids.items():
        print(naam + ": " + str(waarde))

else:
    print("De club is niet gevonden in de database.")

Gevonden identifiers:
Club Naam: Sparta Rotterdam
Short Name: SPA
Team API ID: 8614
Team FIFA ID: 100646
League ID: 13274
Country ID: 13274
Attribute ID: 1187
